# BigBird

#### Это дополнительный ноутбук, более подробно увидеть подход к задаче можно в ноутбуке `LLM_Classification_Finetuning-bert-base.ipynb`
Здесь мы быстренько обучим BigBird, bert-овая модель, работает с последовательностями длины до 4096 токенов. Подробнее [тут](https://huggingface.co/google/bigbird-roberta-large).

In [1]:
import math
import pandas as pd
import torch
from torch.utils import data
from transformers import BigBirdTokenizer, BigBirdForSequenceClassification, Trainer, TrainingArguments

In [3]:
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/data/train.csv')
df

,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie
0,30192,gpt-4-1106-preview,gpt-4-0613,"[""Is it morally right to try to have a certain...","[""The question of whether it is morally right ...","[""As an AI, I don't have personal beliefs or o...",1,0,0
1,53567,koala-13b,gpt-4-0613,"[""What is the difference between marriage lice...","[""A marriage license is a legal document that ...","[""A marriage license and a marriage certificat...",0,1,0
2,65089,gpt-3.5-turbo-0613,mistral-medium,"[""explain function calling. how would you call...","[""Function calling is the process of invoking ...","[""Function calling is the process of invoking ...",0,0,1
3,96401,llama-2-13b-chat,mistral-7b-instruct,"[""How can I create a test set for a very rare ...","[""Creating a test set for a very rare category...","[""When building a classifier for a very rare c...",1,0,0
4,198779,koala-13b,gpt-3.5-turbo-0314,"[""What is the best way to travel from Tel-Aviv...","[""The best way to travel from Tel Aviv to Jeru...","[""The best way to travel from Tel-Aviv to Jeru...",0,1,0
...,...,...,...,...,...,...,...,...,...
57472,4294656694,gpt-4-0613,claude-1,"[""A simple mnemonic for \u03c0:\n\""How I wish ...","[""Sure, let's break it down:\n\n1. \""How\"" has...","[""Here is how that mnemonic represents the dig...",1,0,0
57473,4294692063,claude-2.0,llama-2-13b-chat,"[""In python, implement a naive Bayes with gaus...","[""Here is an implementation of a naive Bayes c...","[""Sure! Here's an implementation of a naive Ba...",1,0,0
57474,4294710549,claude-1,alpaca-13b,"[""is it unethical to work on building weapons?...","[""Working on weapons technology raises some et...","[""It depends on the context. Weapons can be us...",1,0,0
57475,4294899228,palm-2,tulu-2-dpo-70b,"[""If a bait contains 0,0025% bromadiolon then ...","[""Bromadiolone is a rodenticide which is most ...","[""As an AI language model, I do not promote or...",0,1,0


In [4]:
class PairWiseDataset(data.Dataset):
    def __init__(self, df):
        super().__init__()
        self.df = df

    def __getitem__(self, index):
        item = self.df.iloc[index]
        return {
            'id' : int(item['id']),
            'model_a' : item['model_a'],
            'model_b' : item['model_b'],
            'prompt' : item['prompt'],
            'response_a' : item['response_a'],
            'response_b' : item['response_b'],
            'label' : item[['winner_model_a', 'winner_model_b', 'winner_tie']].argmax() if 'winner_model_a' in self.df.columns else None
        }

    def __len__(self):
        return len(self.df)

In [6]:
class BigBirdCollator:
    def __init__(self, tokenizer, max_length=4096, prompt_max_length=1024, pad_to_multiple_of=64, with_labels=True, head_proportion=0.7,):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.prompt_max_length = prompt_max_length
        self.pad_to_multiple_of = pad_to_multiple_of
        self.with_labels = with_labels
        self.head_proportion = head_proportion

    def normalize_text(self, text: str) -> str:
        text = "" if text is None else str(text)

        if text.startswith('["'):
            text = text[2:]
        if text.endswith('"]'):
            text = text[:-2]

        text = text.replace('\\r\\n', '\\n')
        text = text.replace('\\n\\n', '\\n')
        text = text.replace('\\n', ' \\n ')
        text = text.replace('\\u00A0', ' ')
        return text

    def head_and_tail(self, ids, length, head_proportion):
        if len(ids) <= length:
            return ids
        head = int(length * head_proportion)
        tail = length - head
        return ids[:head] + ids[-tail:]

    def tokenize_one(self, prompt, response_a, response_b):
        prompt = self.normalize_text(prompt)
        a = self.normalize_text(response_a)
        b = self.normalize_text(response_b)

        p_ids = self.tokenizer.encode(f"Prompt : {prompt}", add_special_tokens=False)
        a_ids = self.tokenizer.encode(f"Response a : {a}", add_special_tokens=False)
        b_ids = self.tokenizer.encode(f"Response b : {b}", add_special_tokens=False)

        free = self.max_length - 4
        p_len = min(self.prompt_max_length, free)

        remainder = free - p_len
        a_len = remainder // 2
        b_len = remainder - a_len

        p_ids = p_ids[:p_len]
        a_ids = self.head_and_tail(a_ids, a_len, self.head_proportion)
        b_ids = self.head_and_tail(b_ids, b_len, self.head_proportion)

        input_ids = ([self.tokenizer.cls_token_id] + p_ids + [self.tokenizer.sep_token_id] + a_ids + [self.tokenizer.sep_token_id] + b_ids + [self.tokenizer.sep_token_id])

        attention_mask = [1] * len(input_ids)

        return {"input_ids": input_ids, "attention_mask": attention_mask}

    def _ceil_to_multiple(self, x: int, m: int) -> int:
        if m is None or m <= 1:
            return x
        return int(math.ceil(x / m) * m)

    def __call__(self, batch):
        encs = [self.tokenize_one(x["prompt"], x["response_a"], x["response_b"]) for x in batch]

        lengths = [len(e["input_ids"]) for e in encs]
        max_len_in_batch = max(lengths)

        pad_len = self._ceil_to_multiple(max_len_in_batch, self.pad_to_multiple_of)

        pad_id = self.tokenizer.pad_token_id
        input_ids = []
        attention_masks = []

        for e in encs:
            ids = e["input_ids"]
            am = e["attention_mask"]
            pad = pad_len - len(ids)
            if pad > 0:
                ids = ids + [pad_id] * pad
                am = am + [0] * pad
            input_ids.append(ids)
            attention_masks.append(am)

        out = {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_masks, dtype=torch.long),
        }

        if "id" in batch[0]:
            out["id"] = torch.tensor([int(x["id"]) for x in batch], dtype=torch.long)

        if self.with_labels:
            out["labels"] = torch.tensor([int(x["label"]) for x in batch], dtype=torch.long)

        return out

In [29]:
MODEL_NAME = "google/bigbird-roberta-large"
tokenizer = BigBirdTokenizer.from_pretrained(MODEL_NAME)
model = BigBirdForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3, id2label={0: "A", 1: "B", 2: "tie"}, label2id={"A": 0, "B": 1, "tie": 2})

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BigBirdForSequenceClassification LOAD REPORT from: google/bigbird-roberta-large
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.out_proj.bias                   | MISSING    | 
classifier.out_proj.weight                 | MISSING    | 
classifier.dense.weight                    | MISSING    | 
classifier.dense.bias              

In [30]:
ds = PairWiseDataset(df)
train_ds, val_ds = data.random_split(ds, [0.8, 0.2], torch.Generator().manual_seed(33))

collator = BigBirdCollator(tokenizer=tokenizer)

In [32]:
train_args = TrainingArguments(
    output_dir='bigbird_cls',
    per_device_train_batch_size=2,
    num_train_epochs=1,
    learning_rate=4e-5,
    optim='adamw_torch',
    weight_decay=0.01,
    disable_tqdm=False,
    report_to='wandb',
    project='BigBird-LLM-CLS-FT',
    run_name='bb-run1-h100',
    eval_strategy='steps',
    eval_steps=15000,
    eval_delay=10000,
    save_steps=15000,
    per_device_eval_batch_size=2,
    dataloader_num_workers=2,
    warmup_steps=1000,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=1000,
    remove_unused_columns=False,
    gradient_accumulation_steps=1,
    bf16=True
)

trainer = Trainer(
    model=model,
    args=train_args,
    data_collator=collator,
    train_dataset=train_ds,
    eval_dataset=val_ds,
)

model.gradient_checkpointing_enable()

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

model.config.use_cache = False

trainer.train()

Attention type 'block_sparse' is not possible if sequence_length: 448 <= num global tokens: 2 * config.block_size + min. num sliding tokens: 3 * config.block_size + config.num_random_blocks * config.block_size + additional buffer: config.num_random_blocks * config.block_size = 704 with config.block_size = 64, config.num_random_blocks = 3. Changing attention type to 'original_full'...


Step,Training Loss,Validation Loss
15000,1.100203,1.102816


Token indices sequence length is longer than the specified maximum sequence length for this model (4659 > 4096). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (6598 > 4096). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (5269 > 4096). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (7035 > 4096). Running this sequence through the model will result in indexing errors


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Could not locate the best model at bigbird_cls/checkpoint-15000/pytorch_model.bin, if you are running a distributed training on multiple nodes, you should activate `--save_on_each_node`.


TrainOutput(global_step=22991, training_loss=1.103461797466264, metrics={'train_runtime': 4103.8714, 'train_samples_per_second': 11.205, 'train_steps_per_second': 5.602, 'total_flos': 8.85894713354281e+16, 'train_loss': 1.103461797466264, 'epoch': 1.0})

In [31]:
import os, gc, torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
gc.collect()

483